# Cross-Institutional ICL Transfer Analysis

Produces all statistics for Results Section 2.3.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

from src.consolidated_loo_eval import (
    PROJECT_ROOT, MACRO_LABELS, EXPECTED_SEEDS, COMMITTEE_MODEL_NAME,
    discover_variant_specs, load_icl_predictions, merge_with_gold,
    _collapse_model_predictions, _build_seed_level_committee,
    _collapse_committee_predictions, _filter_primary_condition,
    _encode_predictions, bootstrap_macro_f1_ci, paired_bootstrap_comparison,
    _stable_seed, load_gold_labels, ConditionSpec, PROMPT_REGIME_LABEL_ONLY, PROMPT_REGIME_RATIONALE,
)
from sklearn.metrics import precision_recall_fscore_support

PRIMARY_M = 4
BOOTSTRAP_REPS = 5000
ALPHA = 0.05
RANDOM_STATE = 42
OUTPUT_DIR = Path('analysis/cross_institutional_tables')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def disp(df):
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.max_colwidth', None, 'display.width', None):
        display(df)

print(f"PRIMARY_M={PRIMARY_M}, BOOTSTRAP_REPS={BOOTSTRAP_REPS}, ALPHA={ALPHA}")


## Condition Definitions

In [ ]:
CROSS_CONDITIONS = [
    ConditionSpec(key="cross_label_only", source_key="cross_ping",
                  regime="Transfer", prompt_regime=PROMPT_REGIME_LABEL_ONLY, expect_five_seeds=True),
    ConditionSpec(key="cross_rationale", source_key="cross_ping_ra",
                  regime="Transfer", prompt_regime=PROMPT_REGIME_RATIONALE, expect_five_seeds=True),
]
LOCAL_CONDITIONS = [
    ConditionSpec(key="label_only_with_update", source_key="ping",
                  regime="WithUpdate", prompt_regime=PROMPT_REGIME_LABEL_ONLY, expect_five_seeds=True),
    ConditionSpec(key="rationale_with_update", source_key="ping_ra",
                  regime="WithUpdate", prompt_regime=PROMPT_REGIME_RATIONALE, expect_five_seeds=True),
]
COMPARISON_PAIRS = [
    (LOCAL_CONDITIONS[0], CROSS_CONDITIONS[0], "local_label_only vs transfer_label_only"),
    (LOCAL_CONDITIONS[1], CROSS_CONDITIONS[1], "local_rationale vs transfer_rationale"),
]
ALL_CONDITIONS = CROSS_CONDITIONS + LOCAL_CONDITIONS


## Coverage Check

In [ ]:
specs = discover_variant_specs()
print(f"Loaded {len(specs)} variant specs")

coverage_rows = []
for spec in specs:
    gold_df = load_gold_labels(spec.gold_path)
    for cond in CROSS_CONDITIONS:
        raw = load_icl_predictions(spec, cond.source_key)
        merged = merge_with_gold(raw, gold_df) if not raw.empty else raw
        primary = merged[merged["m"].astype(int) == PRIMARY_M] if not merged.empty else merged
        seeds_found = sorted(primary["seed"].dropna().astype(int).unique().tolist()) if not primary.empty else []
        m_found = sorted(raw["m"].dropna().astype(int).unique().tolist()) if not raw.empty else []
        is_complete = seeds_found == sorted(EXPECTED_SEEDS)
        status = "OK" if is_complete else "INCOMPLETE"
        print(f"  [{status}] {spec.cohort}/{spec.prompt_variant}/{cond.source_key}  seeds={seeds_found}")
        coverage_rows.append(dict(cohort=spec.cohort, prompt_variant=spec.prompt_variant,
                                  source_key=cond.source_key, n_rows=len(raw),
                                  seeds=seeds_found, m_values=m_found, complete=is_complete))

coverage_df = pd.DataFrame(coverage_rows)
disp(coverage_df)


## Load & Collapse All Predictions (Local + Transfer)

In [ ]:
prediction_store = {}
table_1_rows = []

for spec in specs:
    gold_df = load_gold_labels(spec.gold_path)
    for cond in ALL_CONDITIONS:
        raw = load_icl_predictions(spec, cond.source_key)
        if raw.empty:
            print(f"  SKIP (empty): {spec.cohort}/{spec.prompt_variant}/{cond.source_key}")
            continue
        merged = merge_with_gold(raw, gold_df)
        primary, is_complete = _filter_primary_condition(
            merged, cond, primary_m=PRIMARY_M, expected_seeds=EXPECTED_SEEDS)
        if primary.empty:
            print(f"  SKIP (no primary): {spec.cohort}/{spec.prompt_variant}/{cond.source_key}")
            continue

        model_preds = _collapse_model_predictions(primary)
        seed_committee = _build_seed_level_committee(primary)
        collapsed_committee = _collapse_committee_predictions(seed_committee)

        for model_name in list(spec.models) + [COMMITTEE_MODEL_NAME]:
            mf = collapsed_committee if model_name == COMMITTEE_MODEL_NAME else \
                 model_preds[model_preds["model"] == model_name].copy()
            if mf.empty:
                continue
            store_key = (spec.cohort, spec.prompt_variant, cond.key, model_name)
            prediction_store[store_key] = mf

            observed, ci_low, ci_high, n_docs = bootstrap_macro_f1_ci(
                mf, B=BOOTSTRAP_REPS, alpha=ALPHA,
                random_state=_stable_seed("cross", spec.cohort, spec.prompt_variant,
                                          cond.key, model_name, base=RANDOM_STATE))
            table_1_rows.append(dict(
                cohort=spec.cohort, prompt_variant=spec.prompt_variant, model=model_name,
                regime=cond.regime, prompt_regime=cond.prompt_regime,
                macro_f1=observed, ci_low=ci_low, ci_high=ci_high, n_docs=n_docs))

transfer_table_1 = pd.DataFrame(table_1_rows)
transfer_table_1.to_csv(OUTPUT_DIR / "transfer_table_1.csv", index=False)
print(f"transfer_table_1: {len(transfer_table_1)} rows")


## Transfer Table 1 - Macro F1 (long prompt)

In [ ]:
t1_long = transfer_table_1[transfer_table_1["prompt_variant"] == "long"].copy()
t1_long["f1_str"] = t1_long.apply(
    lambda r: f"{r['macro_f1']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)

for cohort_name in ["MIMIC", "Indian"]:
    print(f"\n{'='*60}")
    print(f"{cohort_name} cohort - long prompt - m={PRIMARY_M}")
    print(f"{'='*60}")
    sub = t1_long[t1_long["cohort"] == cohort_name].copy()
    pivot = sub.pivot_table(index="model", columns=["regime", "prompt_regime"],
                            values="f1_str", aggfunc="first")
    disp(pivot)


## Transfer Table 1 - Macro F1 (short prompt)

In [ ]:
t1_short = transfer_table_1[transfer_table_1["prompt_variant"] == "short"].copy()
t1_short["f1_str"] = t1_short.apply(
    lambda r: f"{r['macro_f1']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)

for cohort_name in ["MIMIC", "Indian"]:
    print(f"\n{'='*60}")
    print(f"{cohort_name} cohort - short prompt - m={PRIMARY_M}")
    print(f"{'='*60}")
    sub = t1_short[t1_short["cohort"] == cohort_name].copy()
    pivot = sub.pivot_table(index="model", columns=["regime", "prompt_regime"],
                            values="f1_str", aggfunc="first")
    disp(pivot)


## Transfer Table 2 - Per-Class P/R/F1 (long prompt)

In [ ]:
table_2_rows = []
for spec in specs:
    if spec.prompt_variant != "long":
        continue
    gold_df = load_gold_labels(spec.gold_path)
    for cond in ALL_CONDITIONS:
        raw = load_icl_predictions(spec, cond.source_key)
        if raw.empty:
            continue
        merged = merge_with_gold(raw, gold_df)
        primary, _ = _filter_primary_condition(merged, cond,
                                                primary_m=PRIMARY_M, expected_seeds=EXPECTED_SEEDS)
        if primary.empty:
            continue
        model_preds = _collapse_model_predictions(primary)
        seed_committee = _build_seed_level_committee(primary)
        collapsed_committee = _collapse_committee_predictions(seed_committee)

        for model_name in list(spec.models) + [COMMITTEE_MODEL_NAME]:
            mf = collapsed_committee if model_name == COMMITTEE_MODEL_NAME else \
                 model_preds[model_preds["model"] == model_name].copy()
            valid = mf.dropna(subset=["label", "pred"]) if not mf.empty else mf
            if valid.empty:
                continue
            prec, rec, f1, sup = precision_recall_fscore_support(
                valid["label"], valid["pred"], labels=MACRO_LABELS,
                average=None, zero_division=0)
            for i, cls in enumerate(MACRO_LABELS):
                table_2_rows.append(dict(
                    cohort=spec.cohort, model=model_name, regime=cond.regime,
                    prompt_regime=cond.prompt_regime, cls=cls,
                    precision=prec[i], recall=rec[i], f1=f1[i], support=int(sup[i])))

transfer_table_2 = pd.DataFrame(table_2_rows)
transfer_table_2.to_csv(OUTPUT_DIR / "transfer_table_2.csv", index=False)
print(f"transfer_table_2: {len(transfer_table_2)} rows")


### Per-class: rationale-augmented, Local vs Transfer

In [ ]:
for cohort_name in ["MIMIC", "Indian"]:
    print(f"\n{'='*60}")
    print(f"{cohort_name} - long prompt - rationale-augmented - per-class F1")
    print(f"{'='*60}")
    for regime_label in ["WithUpdate", "Transfer"]:
        sub = transfer_table_2[
            (transfer_table_2["cohort"] == cohort_name)
            & (transfer_table_2["regime"] == regime_label)
            & (transfer_table_2["prompt_regime"] == PROMPT_REGIME_RATIONALE)
        ].copy()
        if sub.empty:
            continue
        print(f"\n  --- {regime_label} ---")
        pivot = sub.pivot_table(index="model", columns="cls", values="f1", aggfunc="first")
        pivot["macro_mean"] = pivot.mean(axis=1)
        disp(pivot.round(3))


## Transfer Table 3 - Paired Bootstrap: Local vs Transfer

In [ ]:
table_3_rows = []
for spec in specs:
    for local_cond, cross_cond, comparison_label in COMPARISON_PAIRS:
        for model_name in list(spec.models) + [COMMITTEE_MODEL_NAME]:
            lf = prediction_store.get((spec.cohort, spec.prompt_variant, local_cond.key, model_name))
            cf = prediction_store.get((spec.cohort, spec.prompt_variant, cross_cond.key, model_name))
            stats = paired_bootstrap_comparison(
                lf if isinstance(lf, pd.DataFrame) else pd.DataFrame(),
                cf if isinstance(cf, pd.DataFrame) else pd.DataFrame(),
                B=BOOTSTRAP_REPS, alpha=ALPHA,
                random_state=_stable_seed("cross_compare", spec.cohort, spec.prompt_variant,
                                          comparison_label, model_name, base=RANDOM_STATE))
            table_3_rows.append(dict(
                cohort=spec.cohort, prompt_variant=spec.prompt_variant, model=model_name,
                comparison=comparison_label,
                delta_f1=stats["delta_f1"],
                ci_low=stats["bootstrap_ci_low"], ci_high=stats["bootstrap_ci_high"],
                p=stats["bootstrap_p"],
                sig=bool(stats["bootstrap_p"] < ALPHA) if pd.notna(stats["bootstrap_p"]) else False))

transfer_table_3 = pd.DataFrame(table_3_rows)
transfer_table_3.to_csv(OUTPUT_DIR / "transfer_table_3.csv", index=False)
print(f"transfer_table_3: {len(transfer_table_3)} rows")


### Paired bootstrap results (long prompt)

In [ ]:
t3_long = transfer_table_3[transfer_table_3["prompt_variant"] == "long"].copy()

for cohort_name in ["MIMIC", "Indian"]:
    print(f"\n{'='*60}")
    print(f"{cohort_name} - long prompt - delta = local - transfer (positive = local better)")
    print(f"{'='*60}")
    sub = t3_long[t3_long["cohort"] == cohort_name].copy()
    sub["result"] = sub.apply(
        lambda r: f"delta={r['delta_f1']:+.3f} [{r['ci_low']:+.3f},{r['ci_high']:+.3f}] p={r['p']:.4f}"
                  + (" *" if r["sig"] else ""), axis=1)
    pivot = sub.pivot_table(index="model", columns="comparison", values="result", aggfunc="first")
    disp(pivot)


### Paired bootstrap results (short prompt)

In [ ]:
t3_short = transfer_table_3[transfer_table_3["prompt_variant"] == "short"].copy()

for cohort_name in ["MIMIC", "Indian"]:
    print(f"\n{'='*60}")
    print(f"{cohort_name} - short prompt - delta = local - transfer (positive = local better)")
    print(f"{'='*60}")
    sub = t3_short[t3_short["cohort"] == cohort_name].copy()
    sub["result"] = sub.apply(
        lambda r: f"delta={r['delta_f1']:+.3f} [{r['ci_low']:+.3f},{r['ci_high']:+.3f}] p={r['p']:.4f}"
                  + (" *" if r["sig"] else ""), axis=1)
    pivot = sub.pivot_table(index="model", columns="comparison", values="result", aggfunc="first")
    disp(pivot)


## Transfer Table 4 - Aggregate Summary

In [ ]:
t3_models = transfer_table_3[transfer_table_3["model"] != COMMITTEE_MODEL_NAME].copy()

summary_rows = []
for (cohort, pv, comp), grp in t3_models.groupby(["cohort", "prompt_variant", "comparison"]):
    n = len(grp)
    n_sig = grp["sig"].sum()
    n_local_better = ((grp["delta_f1"] > 0) & grp["sig"]).sum()
    n_transfer_better = ((grp["delta_f1"] < 0) & grp["sig"]).sum()
    summary_rows.append(dict(
        cohort=cohort, prompt_variant=pv, comparison=comp, n_models=n,
        mean_delta=grp["delta_f1"].mean(), median_delta=grp["delta_f1"].median(),
        mean_abs_delta=grp["delta_f1"].abs().mean(), max_abs_delta=grp["delta_f1"].abs().max(),
        n_nonsig=int(n - n_sig), n_local_sig_better=int(n_local_better),
        n_transfer_sig_better=int(n_transfer_better)))

transfer_table_4 = pd.DataFrame(summary_rows)
transfer_table_4.to_csv(OUTPUT_DIR / "transfer_table_4.csv", index=False)
disp(transfer_table_4)


## Per-Class Degradation Analysis

In [ ]:
for cohort_name in ["MIMIC", "Indian"]:
    print(f"\n{'='*60}")
    print(f"{cohort_name} - per-class delta F1 (local - transfer), rationale, long prompt")
    print(f"{'='*60}")

    local_sub = transfer_table_2[
        (transfer_table_2["cohort"] == cohort_name)
        & (transfer_table_2["regime"] == "WithUpdate")
        & (transfer_table_2["prompt_regime"] == PROMPT_REGIME_RATIONALE)
    ]
    transfer_sub = transfer_table_2[
        (transfer_table_2["cohort"] == cohort_name)
        & (transfer_table_2["regime"] == "Transfer")
        & (transfer_table_2["prompt_regime"] == PROMPT_REGIME_RATIONALE)
    ]
    if local_sub.empty or transfer_sub.empty:
        print("  Insufficient data"); continue

    merged = local_sub.merge(transfer_sub, on=["cohort", "model", "cls"],
                              suffixes=("_local", "_transfer"), how="inner")
    merged["delta_f1"] = merged["f1_local"] - merged["f1_transfer"]

    # Per model per class
    print("\n  Per-model detail:")
    detail = merged[merged["model"] != COMMITTEE_MODEL_NAME][
        ["model", "cls", "f1_local", "f1_transfer", "delta_f1"]
    ].copy()
    detail["model_short"] = detail["model"].apply(lambda x: x.split("/")[-1] if "/" in str(x) else x)
    pivot = detail.pivot_table(index="model_short", columns="cls", values="delta_f1", aggfunc="first")
    disp(pivot.round(3))

    # Aggregate per class
    print("\n  Averaged across 7 models:")
    agg = detail.groupby("cls")["delta_f1"].agg(["mean", "median", "min", "max"]).round(3)
    disp(agg)


## Rationale Portability: Label-only vs Rationale under Transfer

In [ ]:
print("Does rationale augmentation help / hurt under cross-institutional transfer?")
print("Comparing transfer_label_only vs transfer_rationale for each model.\n")

for cohort_name in ["MIMIC", "Indian"]:
    print(f"--- {cohort_name} (long prompt) ---")
    rows = []
    for spec in specs:
        if spec.cohort != cohort_name or spec.prompt_variant != "long":
            continue
        for model_name in list(spec.models) + [COMMITTEE_MODEL_NAME]:
            lo = prediction_store.get((cohort_name, "long", "cross_label_only", model_name))
            ra = prediction_store.get((cohort_name, "long", "cross_rationale", model_name))
            stats = paired_bootstrap_comparison(
                ra if isinstance(ra, pd.DataFrame) else pd.DataFrame(),
                lo if isinstance(lo, pd.DataFrame) else pd.DataFrame(),
                B=BOOTSTRAP_REPS, alpha=ALPHA,
                random_state=_stable_seed("ra_port", cohort_name, model_name, base=RANDOM_STATE))
            model_short = model_name.split("/")[-1] if "/" in str(model_name) else model_name
            rows.append(dict(model=model_short,
                             delta=stats["delta_f1"], ci_lo=stats["bootstrap_ci_low"],
                             ci_hi=stats["bootstrap_ci_high"], p=stats["bootstrap_p"],
                             sig=bool(stats["bootstrap_p"] < ALPHA) if pd.notna(stats["bootstrap_p"]) else False))

    rdf = pd.DataFrame(rows)
    rdf["summary"] = rdf.apply(
        lambda r: f"delta={r['delta']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] p={r['p']:.4f}"
                  + (" *" if r["sig"] else ""), axis=1)
    print("  delta = rationale - label_only (positive = rationale better)")
    disp(rdf[["model", "summary"]])
    print()


## Complete Numeric Dump 

In [ ]:
print("="*70)
print("ALL TRANSFER F1 VALUES (for copy-paste into LaTeX)")
print("="*70)

for pv in ["long", "short"]:
    for cohort_name in ["MIMIC", "Indian"]:
        print(f"\n--- {cohort_name} / {pv} ---")
        sub = transfer_table_1[
            (transfer_table_1["cohort"] == cohort_name)
            & (transfer_table_1["prompt_variant"] == pv)
        ].sort_values(["regime", "prompt_regime", "model"])
        for _, r in sub.iterrows():
            m = r["model"].split("/")[-1] if "/" in str(r["model"]) else r["model"]
            print(f"  {r['regime']:12s} {r['prompt_regime']:30s} {m:40s} "
                  f"{r['macro_f1']:.4f} [{r['ci_low']:.4f}, {r['ci_high']:.4f}]")

print("\n" + "="*70)
print("ALL PAIRED BOOTSTRAP DELTAS (for copy-paste)")
print("="*70)

for pv in ["long", "short"]:
    for cohort_name in ["MIMIC", "Indian"]:
        print(f"\n--- {cohort_name} / {pv} ---")
        sub = transfer_table_3[
            (transfer_table_3["cohort"] == cohort_name)
            & (transfer_table_3["prompt_variant"] == pv)
        ].sort_values(["comparison", "model"])
        for _, r in sub.iterrows():
            m = r["model"].split("/")[-1] if "/" in str(r["model"]) else r["model"]
            sig = " *" if r["sig"] else ""
            print(f"  {r['comparison']:50s} {m:40s} "
                  f"delta={r['delta_f1']:+.4f} [{r['ci_low']:+.4f},{r['ci_high']:+.4f}] "
                  f"p={r['p']:.4f}{sig}")

print("\nDone. Upload this notebook output to proceed with section 2.3 writeup.")
